  <img src="../images/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Ciência de Dados EaD - 2026/02**


# **Documentação do Projeto**

---


# **Paridade de preços de combustíveis, atividade econômica e a ação da Petrobras**
---


In [ ]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: \<nome\>, \<matrícula\>*)
#@markdown AVA: grupo 19
Aluno1 = 'Guilherme Augusto Leal Oliveira, 10730632' #@param {type:"string"}
Aluno2 = 'Guilherme Rocha de Souza Duarte, 10433651' #@param {type:"string"}
Aluno3 = 'Ricardo Zulian de Souza Amaral' #@param {type:"string"}
Aluno4 = '' #@param {type:"string"}
Aluno5 = '' #@param {type:"string"}


# **Resumo**

O preço doméstico de diesel e gasolina no Brasil se afasta com frequência da paridade internacional. Esse desvio, medido pelo PPI da ABICOM, altera custo de frete, mobilidade nas cidades e a cotação da Petrobras. O grupo constrói um produto de série temporal que estima a defasagem entre choques de paridade e três alvos: o retorno de PETR4, o volume de combustível e indicadores de tráfego, atividade e emprego. A base é um snapshot congelado de séries oficiais (ABICOM, ANP, Petrobras, B3, ABCR, IBGE e BCB) em SQLite, com âncora diária no PPI de 5 de janeiro de 2022 a 14 de agosto de 2026. A metodologia prevista alinha frequências mistas e aplica modelos estatísticos de série temporal com variáveis exógenas. O produto apoia a leitura da política de preços e dos efeitos sobre trabalho, cidades e instituições (ODS 8, 11 e 16). Os resultados numéricos entram nas etapas 3 e 4.


# **Introdução**

## Contexto

O Brasil combina preço na refinaria com a paridade internacional como referência de mercado. O PPI (Índice de Paridade de Preços) da ABICOM publica, em dias úteis, o diferencial em R$/L e em % entre o preço Petrobras e a paridade de importação para gasolina e diesel (ABICOM, 2026). Diesel move carga. Gasolina move o tráfego leve nas cidades. A Petrobras é empresa de economia mista listada na B3; o papel PETR4 reage a reajuste e a paridade. O tema é série temporal com frequências mistas: dia, semana, mês e trimestre (SILVA; PERES; BOSCARIOLI, 2016; HYNDMAN; ATHANASOPOULOS, 2021).

O recorte liga três Objetivos de Desenvolvimento Sustentável (ONU, 2024). O ODS 8 trata de trabalho e crescimento: diesel, tráfego pesado, indústria e desemprego. O ODS 11 trata de cidades: gasolina, tráfego leve e preço ao consumidor. O ODS 16 trata de instituições: a política de preços de uma empresa estatal de capital aberto e o PPI como sinal público do desvio em relação à paridade.

## Motivação

A política de preços da Petrobras mudou mais de uma vez nesta década. O mercado observa o PPI, mas a transmissão até o posto, o volume e o emprego não são imediatas nem uniformes. Existe uma lacuna entre o cartão diário de paridade e uma medida reproduzível de defasagem até atividade, desemprego e tráfego. As fontes públicas existem (ANP, IBGE, ABCR, B3, BCB), porém em calendários distintos. O grupo junta essas séries em um recorte único a partir de 2022 e testa se o PPI antecipa preço, volume, tráfego e o retorno de PETR4.

## Objetivo

O estudo estima a transmissão temporal de choques de paridade e de preço de combustível no Brasil. O caminho vai da refinaria até a ação PETR4, o volume, o tráfego e os indicadores de atividade e emprego.

Objetivos específicos:

1. Organizar a base congelada com âncora no PPI diário.
2. Medir defasagens entre PPI, preço ANP/Petrobras e PETR4.
3. Testar se choques de diesel antecipam tráfego pesado, indústria e desemprego (ODS 8).
4. Testar se choques de gasolina e tráfego leve descrevem mobilidade urbana (ODS 11).
5. Documentar o PPI como sinal público da política de preços da Petrobras (ODS 16).
6. Entregar um notebook reproduzível com modelo estatístico de série temporal.

A hipótese de trabalho é que o choque de paridade e de preço passa primeiro pela ação e, depois, pela indústria, pelo varejo, pelo emprego e pelo tráfego.

## Justificativa

A contribuição analítica é um recorte reproduzível de fontes originais, sem base pré-processada de terceiro, com frequência mista e âncora diária. A contribuição prática são defasagens úteis para quem lê política de preços, frete e mobilidade. O ODS 8 liga diesel, emprego e crescimento. O ODS 11 liga gasolina, tráfego leve e cidades. O ODS 16 liga transparência da política de preços e da Petrobras como instituição. O produto é um experimento executável, não um painel de produção.

## Descrição da base de dados

A base do curso é o arquivo `data/gas_flare.sqlite`. O arquivo é um snapshot congelado. O projeto não acessa Postgres, API ao vivo nem credencial. As séries vêm de fontes originais (governo e entidades setoriais), reunidas em 14 tabelas. A âncora temporal é o PPI diário: 5 de janeiro de 2022 a 14 de agosto de 2026, com 1.126 datas (ABICOM, 2026). As demais famílias entram nesse calendário de dias úteis, cada uma com a defasagem da fonte.

Quadro 1 — Famílias da base congelada

| Família | Frequência | Conteúdo | Recorte no snapshot | Fonte |
|---|---|---|---|---|
| PPI | diária | diferencial Petrobras e independentes vs. paridade, gasolina e diesel, R$/L e % | 5 jan. 2022 a 14 ago. 2026 (1.126 datas) | ABICOM |
| Ação PETR4 | diária (pregão) | abertura, máxima, mínima, fechamento, volume | 1º fev. 2022 a 18 ago. 2026 (1.134 pregões) | B3 |
| Preço ANP | semanal | média ao produtor/importação, gasolina e diesel, R$/L | 3 jan. 2022 a 9 ago. 2026 (240 semanas por produto) | ANP |
| Preço Petrobras | semanal irregular | preço às distribuidoras, R$/L | gasolina até maio/2026; diesel até ago./2026 | Petrobras |
| Vendas ANP | mensal | volume nacional, m³ | jan. 2022 a jun. 2026 | ANP |
| Vendas e finanças Petrobras | trimestral | volume (mbpd), receita e margem RT&M | vendas 2022T1–2026T2; finanças desde 2006T1 | Petrobras |
| Tráfego | mensal | índice ABCR leve, pesado e total | fev. 2022 a jul. 2026 | ABCR |
| Atividade e emprego | mensal | desemprego (PNAD), varejo (PMC), indústria (PIM-PF) | jan. 2022 a jun. 2026 | IBGE |
| Inflação | mensal | IPCA, IPCA-15 e IGP-M | jan. 2022 a jul. 2026 | IBGE / FGV via BCB |

A coleta operacional de origem é semanal. O curso usa o estado congelado. Semanas sem PDF novo da Petrobras não avançam a série de preço à distribuidora: isso é ausência de reajuste, não falha de carga. PETR4 começa em fevereiro de 2022; os NaNs de janeiro no índice do PPI são esperados. Os preços da B3 não têm ajuste por proventos. O dicionário das tabelas está em `docs/metadados_base.md`.


# **Referencial Teórico**

*(Etapa 2 — máximo 1 página.)* Definir os conceitos da solução. Discutir trabalhos correlatos. Apresentar alternativas e limites. Citar no padrão ABNT.


# **Diagrama de Solução**

*(Etapa 2.)* Apresentar e discutir o diagrama da solução proposta.


# **EDA e Pré-processamento dos dados**

*(Etapa 3.)* Exploração e análise dos dados. Qualidade, limites e recortes. Transformações, junções e encodes.


In [ ]:
# Códigos aqui (etapa 3)


# **Modelos**

*(Etapas 3 e 4.)* Descrição metodológica da criação, do refinamento e da validação dos modelos.


In [ ]:
# Códigos aqui (etapas 3 e 4)


# **Resultados**

*(Etapa 4.)* Métricas, tabelas comparativas e leitura dos resultados.


# **Discussão e Conclusão**

*(Etapa 4.)* Qualidades e limites do trabalho. Alcance do objetivo. Melhorias possíveis.


# **Apresentação**

*(Etapa 4.)* Link do vídeo (apresentação com slides).


# **Referências**

ASSOCIAÇÃO BRASILEIRA DE CONCESSIONÁRIAS DE RODOVIAS. *Índice ABCR*. São Paulo: ABCR, 2026. Disponível em: https://melhoresrodovias.org.br/. Acesso em: 19 ago. 2026.

ASSOCIAÇÃO BRASILEIRA DOS IMPORTADORES DE COMBUSTÍVEIS. *PPI*. Rio de Janeiro: ABICOM, 2026. Disponível em: https://abicom.com.br/categoria/ppi/. Acesso em: 19 ago. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Dados abertos*: Série Histórica do Levantamento de Preços e de Margens de Comercialização de Combustíveis (SHPC). Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/. Acesso em: 19 ago. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Vendas de combustíveis (m³)*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos. Acesso em: 19 ago. 2026.

BANCO CENTRAL DO BRASIL. *Sistema Gerenciador de Séries Temporais (SGS)*. Brasília: BCB, 2026. Disponível em: https://www3.bcb.gov.br/sgspub/. Acesso em: 19 ago. 2026.

B3 S.A. – BRASIL, BOLSA, BALCÃO. *Cotações históricas*. São Paulo: B3, 2026. Disponível em: https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/cotacoes-historicas/. Acesso em: 19 ago. 2026.

HYNDMAN, R. J.; ATHANASOPOULOS, G. *Forecasting*: principles and practice. 3. ed. Melbourne: OTexts, 2021. Disponível em: https://otexts.com/fpp3/. Acesso em: 24 ago. 2026.

INSTITUTO BRASILEIRO DE GEOGRAFIA E ESTATÍSTICA. *SIDRA*: Sistema IBGE de Recuperação Automática. Rio de Janeiro: IBGE, 2026. Disponível em: https://sidra.ibge.gov.br/. Acesso em: 19 ago. 2026.

ORGANIZAÇÃO DAS NAÇÕES UNIDAS. *Sustainable Development Goals*. Nova York: ONU, 2024. Disponível em: https://www.undp.org/sustainable-development-goals. Acesso em: 24 ago. 2026.

PETROBRAS. *Central de resultados*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://www.investidorpetrobras.com.br/resultados-e-comunicados/central-de-resultados/. Acesso em: 19 ago. 2026.

PETROBRAS. *Tabelas de preços de venda às distribuidoras*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://precos.petrobras.com.br. Acesso em: 19 ago. 2026.

SILVA, L. A.; PERES, S. M.; BOSCARIOLI, C. *Introdução à mineração de dados com aplicações em R*. 1. ed. São Paulo: LTC, 2016.
